In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import asyncio
import time as _time
import re
import os
import pandas as pd
from tqdm.notebook import tqdm
from openai import AsyncOpenAI

from lib.llm.llm_client import (
    MODEL,
    MODEL_N,
    timed_completion,
    strip_thinking,
    compute_cost,
)

_async_client = None

def _get_async_client():
    global _async_client
    if _async_client is None:
        _async_client = AsyncOpenAI(
            api_key=os.getenv(f"LLM_API_KEY_{MODEL_N}"),
            base_url=os.getenv(f"LLM_BASE_URL_{MODEL_N}"),
        )
    return _async_client


In [3]:
# ── Verify thinking is disabled ─────────────────────────────────────────
from lib.llm.llm_client import _THINKING_OFF_EXTRA_BODY

_method = 'extra_body' if _THINKING_OFF_EXTRA_BODY.get(MODEL_N) else 'none (thinking hardcoded by server)'
print(f'Model    : MODEL_{MODEL_N} = {MODEL}')
print(f'Method   : {_method}')
print()

_test_response, _test_latency = timed_completion(
    messages=[{'role': 'user', 'content': 'Reply with the single word: hello'}],
    model=MODEL,
    n=MODEL_N,
    temperature=0,
)
_raw = _test_response.choices[0].message.content
_has_thinking = '<think>' in _raw
print('Thinking disabled:', not _has_thinking)
print('Raw response     :', repr(_raw[:100]))
if _has_thinking:
    print()
    print('NOTE: thinking cannot be disabled on this server — strip_thinking() removes it from all outputs.')


Model    : MODEL_1 = UTM2
Method   : extra_body

Thinking disabled: True
Raw response     : 'hello'


In [4]:
df = pd.read_csv("../../datasets/esgenius_perturbed.csv")

print(df.shape)
df.head()

(21584, 14)


,query_id,new_id,query,answer,A,B,C,D,Z,ref_page,ref_doc,source_text,perturbation_type,perturbation_level
0,831,1,"According to the IPCC AR6 Synthesis Report, wh...",A,Implementing targeted management strategies fo...,Failing to rebuild overexploited fisheries whi...,Limiting global warming but neglecting land re...,Prioritizing disaster risk management and earl...,Not sure,46,IPCC AR6 SYR.pdf,30 Summary for Policymakers Summary for Policy...,none,0
1,889,2,According to Climate Change 2023 — Synthesis R...,B,The geographical patterns of climatic impacts ...,Geographical patterns of climatic impacts for ...,The timing of reaching a specific GWL determin...,Global warming levels are defined exclusively ...,Not sure,80-81,IPCC AR6 SYR.pdf,64 Section 2 Section 1Section 2Global Warming ...,none,0
2,892,3,Which statement accurately reflects the relati...,C,High or very high confidence in attribution is...,The text explicitly states that medium confide...,Attribution of observed physical climate chang...,Increases in heavy precipitation and sea level...,Not sure,23,IPCC AR6 SYR.pdf,7 Summary for PolicymakersSummary for Policyma...,none,0
3,983,4,According to the Climate Change 2023 — Synthes...,D,Europe,Small Islands,North America,Asia,Not sure,92,IPCC AR6 SYR.pdf,76 Section 3 Section 1Section 3011.5234 011.52...,none,0
4,1077,5,According to the Climate Change 2023 — Synthes...,A,"Early warning systems, flood-proofing of build...","Afforestation, reforestation, and levees.","Agroecological practices, disaster risk manage...","Heat Health Action Plans, vector-borne disease...",Not sure,72,IPCC AR6 SYR.pdf,"56 Section 2 Section 1Section 2wetlands, range...",none,0


In [5]:
PROMPT_TEMPLATE = """
You are answering a multiple-choice question.

Use ONLY the provided source text.

Question:
{query}

Options:
A. {A}

B. {B}

C. {C}

D. {D}

Z. {Z}

Source Text:
{source_text}

Return ONLY one letter:
A
B
C
D
or Z
"""

In [6]:
VALID_ANSWERS = {"A", "B", "C", "D", "Z"}

async def predict_answer_async(row):
    prompt = PROMPT_TEMPLATE.format(
        query=row["query"],
        A=row["A"], B=row["B"], C=row["C"], D=row["D"], Z=row["Z"],
        source_text=row["source_text"],
    )
    messages = [{"role": "user", "content": prompt}]
    try:
        client = _get_async_client()
        t0 = _time.perf_counter()
        response = await client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            timeout=60,
        )
        latency_ms = (_time.perf_counter() - t0) * 1000
        text = strip_thinking(response.choices[0].message.content).strip()
        match = re.search(r"\b([ABCDZ])\b", text)
        return (match.group(1) if match else "INVALID"), latency_ms, response
    except Exception as e:
        print(e)
        return "ERROR", 0.0, None


In [7]:
from tqdm import tqdm as tqdm_sync

MAX_CONCURRENT = 35

async def predict_row_async(row, semaphore):
    async with semaphore:
        guess, latency, response = await predict_answer_async(row)
        return {
            "query_id":           row["query_id"],
            "perturbation_type":  row["perturbation_type"],
            "perturbation_level": row["perturbation_level"],
            "ground_truth":       row["answer"],
            "llm_guess":          guess,
            "llm_latency":        latency,
            "prompt_tokens":      response.usage.prompt_tokens     if response else None,
            "completion_tokens":  response.usage.completion_tokens if response else None,
            "cost_usd":           compute_cost(response, MODEL_N)  if response else None,
        }

async def run_classification(df):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    rows = df.to_dict("records")
    tasks = [predict_row_async(row, semaphore) for row in rows]
    out = []
    for coro in tqdm_sync(asyncio.as_completed(tasks), total=len(tasks)):
        out.append(await coro)
    return out

predictions = await run_classification(df)
results_df = pd.DataFrame(predictions)
results_df.head()


100%|██████████| 21584/21584 [1:07:35<00:00,  5.32it/s]


,query_id,perturbation_type,perturbation_level,ground_truth,llm_guess,llm_latency,prompt_tokens,completion_tokens,cost_usd
0,12914,char,1,C,C,2275.761875,1048,2,0.000159
1,12910,char,1,A,A,3397.777042,1450,2,0.000219
2,16717,numeric,1,D,D,3396.906041,1239,2,0.000188
3,16714,numeric,1,A,A,3401.984375,1144,2,0.000173
4,16532,label,1,A,D,3411.773125,798,2,0.000122


In [8]:
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path("../../datasets/experiments/llm/classification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = OUTPUT_DIR / f"esgenius_classification_{timestamp}.csv"

results_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE.resolve()}")


Saved to: /Users/fabin/Documents/GitHub/llmb-project-group-1/datasets/experiments/llm/classification/esgenius_classification_20260625_172036.csv
